# SAM — IoT Environmental Data Preprocessing & Merge

**Pipeline:**
1. Load `sensor_data` (10-min), `tmd_data` (3-hr), `aqi_data` (1-hr)
2. Clean rows where sensor failure produced invalid `0` values
3. Sort all tables by timestamp
4. Merge with `merge_asof(direction='nearest')` — keeps all primary rows
5. Save → `output/integrated_air_quality_data.csv`

## 0 · Imports

In [19]:
import os
import pymysql
import pandas as pd
import numpy as np
from dotenv import load_dotenv

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

## 1 · Configuration

### 1a · File paths
Point these at your exported CSV files (or swap the Load section below for a direct MySQL query).

In [20]:
# ── Input files ──────────────────────────────────────────────────────────────
# Export these from MySQL / phpMyAdmin as UTF-8 CSV before running
SENSOR_DATA_CSV = 'data/sensor_data.csv'   # primary — every ~10 min
TMD_DATA_CSV    = 'data/tmd_data.csv'      # secondary — every ~3 hr
AQI_DATA_CSV    = 'data/aqi_data.csv'      # secondary — every ~1 hr

# ── Output ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = 'output'
OUTPUT_CSV = os.path.join(OUTPUT_DIR, 'integrated_air_quality_data.csv')

# ── Timestamp column name (same in all three tables) ─────────────────────────
TS_COL = 'ts'

### 1b · Zero-value cleaning config

Sensors occasionally fail and return `0` instead of a real reading.

**How to use:**
- Add a column name to `ZERO_INVALID_COLS` if a `0` in that column means a *bad reading* (the whole row will be dropped).
- Remove a column from the list if `0` is a *valid* value (e.g., `gas_smoke` during clean-air baseline — unlikely but possible).
- Columns not in this list are **never** dropped on a zero — keeps the data.

In [21]:
# Columns where 0 = sensor failure → drop the entire row
# Edit this list freely. Order does not matter.
ZERO_INVALID_COLS: list[str] = [
    'pm25',       # dust sensor hardware failure always reads 0
    'temp',       # DHT/KY sensor dropout
    'humidity',   # paired with temp — 0 means same dropout
    # 'gas_smoke',  # uncomment if smoke sensor also fails to 0
    'gas_co'     # uncomment if CO sensor also fails to 0
]

# ── Column name mapping ───────────────────────────────────────────────────────
# Map the raw DB column names → friendlier analysis names.
# Keys = column names AS THEY APPEAR IN YOUR CSV.
# Values = what you want them called in the merged output.
# Set to {} to skip renaming.
SENSOR_RENAME: dict[str, str] = {
    'temp_dht':  'temp',        # use DHT temperature as primary temp
    'co_raw':    'gas_co',      # raw CO ADC reading
    'smoke_raw': 'gas_smoke',   # raw smoke ADC reading
    # 'temp_ky': 'temp_ky',     # keep if you want the second sensor too
}

# Columns to keep from each table (after rename).
# None = keep all columns.
SENSOR_KEEP_COLS = ['ts', 'temp', 'humidity', 'pm25', 'pm10', 'gas_co', 'gas_smoke', 'place', 'lat', 'lon']
TMD_KEEP_COLS    = ['ts', 'temp', 'humidity', 'rainfall', 'lat', 'lon']
AQI_KEEP_COLS    = ['ts', 'pm25', 'pm10', 'lat', 'lon']

## 2 · Helper functions

In [22]:
def load_table(path: str, ts_col: str = TS_COL, rename: dict | None = None,
               keep_cols: list | None = None) -> pd.DataFrame:
    """Load a CSV, parse timestamps, optionally rename and select columns."""
    df = pd.read_csv(path, parse_dates=[ts_col])
    if rename:
        df = df.rename(columns=rename)
    if keep_cols:
        # Only keep columns that actually exist (graceful for optional fields)
        existing = [c for c in keep_cols if c in df.columns]
        df = df[existing]
    print(f'  Loaded {path!r}: {len(df):,} rows, columns: {list(df.columns)}')
    return df


def drop_zero_rows(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """
    Drop rows where any column in `cols` equals exactly 0.

    Columns not in `cols` are untouched — a 0 there is kept.
    Only drops on columns that actually exist in the dataframe.
    """
    before = len(df)
    active_cols = [c for c in cols if c in df.columns]
    if not active_cols:
        print('  No matching zero-clean columns found — nothing dropped.')
        return df

    # Build a mask: True where any active column == 0
    zero_mask = (df[active_cols] == 0).any(axis=1)
    df_clean = df[~zero_mask].copy()

    dropped = before - len(df_clean)
    pct = dropped / before * 100 if before else 0
    print(f'  Zero-clean on {active_cols}: dropped {dropped:,} rows ({pct:.1f}%) → {len(df_clean):,} remaining')
    return df_clean


def sort_by_ts(df: pd.DataFrame, ts_col: str = TS_COL) -> pd.DataFrame:
    """Sort by timestamp and reset index."""
    return df.sort_values(ts_col).reset_index(drop=True)


def suffix_cols(df: pd.DataFrame, suffix: str, exclude: list[str]) -> pd.DataFrame:
    """Rename columns with a suffix, except those in `exclude`."""
    return df.rename(columns={
        c: f'{c}{suffix}' for c in df.columns if c not in exclude
    })

## 3 · Load data

In [23]:
import warnings

load_dotenv('../backend/.env')

DB_HOST   = os.getenv('DB_HOST', 'localhost')
DB_USER   = os.getenv('DB_USER', 'root')
DB_PASSWD = os.getenv('DB_PASSWORD', '')
DB_NAME   = os.getenv('DB_NAME', 'sam')

sensor = tmd = aqi = None

try:
    with pymysql.connect(
        host=DB_HOST,
        user=DB_USER,
        password=DB_PASSWD,
        database=DB_NAME,
    ) as connection:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', UserWarning)
            sensor = pd.read_sql('SELECT * FROM sensor_data ORDER BY ts', connection)
            tmd    = pd.read_sql('SELECT * FROM tmd_data   ORDER BY ts', connection)
            aqi    = pd.read_sql('SELECT * FROM aqi_data   ORDER BY ts', connection)
        print('Loaded from MySQL — '
              f'sensor: {len(sensor):,} | tmd: {len(tmd):,} | aqi: {len(aqi):,}')
except Exception as e:
    print(f'Error: {e}')

# Apply rename + column filter from config
if sensor is not None:
    sensor = sensor.rename(columns=SENSOR_RENAME)
    sensor = sensor[[c for c in SENSOR_KEEP_COLS if c in sensor.columns]]
    tmd    = tmd[[c for c in TMD_KEEP_COLS    if c in tmd.columns]]
    aqi    = aqi[[c for c in AQI_KEEP_COLS    if c in aqi.columns]]

Loaded from MySQL — sensor: 609 | tmd: 139 | aqi: 429


## 4 · Inspect raw data

In [24]:
print('=== sensor_data sample ===')
display(sensor.head(3))
display(sensor.dtypes)
display(sensor.describe())

=== sensor_data sample ===


,ts,temp,humidity,pm25,pm10,gas_co,gas_smoke,place,lat,lon
0,2026-03-31 00:55:18,29.0000,58,13,10,282,819,inside,13.8463,100.5680
1,2026-03-31 00:55:35,27.0000,45,16,12,277,29,inside,13.8463,100.5680
2,2026-03-31 01:05:35,26.0000,43,16,12,291,561,inside,13.8463,100.5680


ts           datetime64[ns]
temp                float64
humidity              int64
pm25                  int64
pm10                  int64
gas_co                int64
gas_smoke             int64
place                object
lat                 float64
lon                 float64
dtype: object

,ts,temp,humidity,pm25,pm10,gas_co,gas_smoke,lat,lon
count,609,609.0000,609.0000,609.0000,609.0000,609.0000,609.0000,609.0000,609.0000
mean,2026-04-04 18:36:24.737274112,30.8473,51.7504,21.7077,15.0378,218.6010,272.4893,13.8463,100.5680
min,2026-03-31 00:55:18,25.0000,35.0000,0.0000,0.0000,0.0000,0.0000,13.8463,100.5680
25%,2026-04-01 03:46:41,27.0000,47.0000,0.0000,0.0000,186.0000,0.0000,13.8463,100.5680
50%,2026-04-06 23:25:28,29.0000,51.0000,25.0000,17.0000,221.0000,119.0000,13.8463,100.5680
75%,2026-04-08 00:38:51,35.0000,57.0000,34.0000,24.0000,251.0000,437.0000,13.8463,100.5680
max,2026-04-09 08:45:34,39.0000,73.0000,53.0000,36.0000,320.0000,2303.0000,13.8463,100.5680
std,NaN,4.0113,7.3112,17.1624,11.5909,40.4159,340.0195,0.0000,0.0000


In [25]:
print('=== tmd_data sample ===')
display(tmd.head(3))

print('=== aqi_data sample ===')
display(aqi.head(3))

=== tmd_data sample ===


,ts,temp,humidity,rainfall,lat,lon
0,2026-03-31 10:00:00,32.5000,58.0000,0.0000,13.1617,100.8020
1,2026-03-31 19:00:00,32.3000,53.0000,0.0000,13.9192,100.6050
2,2026-03-31 22:00:00,28.7000,80.0000,0.0000,13.4893,99.7924


=== aqi_data sample ===


,ts,pm25,pm10,lat,lon
0,2026-03-31 01:00:00,63,16,13.8476,100.5790
1,2026-03-31 01:00:00,63,16,13.8476,100.5790
2,2026-03-31 01:00:00,63,16,13.8476,100.5790


## 5 · Clean: drop zero-value sensor failures

Uses `ZERO_INVALID_COLS` defined in §1b.  
Only rows in the **primary** `sensor_data` table are cleaned here.  
TMD and AQI data are weather/reference data — zeros there may be valid (e.g., `rainfall=0`).

In [26]:
print('Cleaning sensor_data zeros...')
sensor_clean = drop_zero_rows(sensor, ZERO_INVALID_COLS)
sensor_clean = sensor_clean.drop(columns=['gas_smoke'])
print(sensor_clean.shape[0])
# -- Optional: also clean tmd/aqi if needed --
# Uncomment and set cols if TMD/AQI also have sensor-failure zeros:
# tmd_clean = drop_zero_rows(tmd, ['temp'])  # example
# aqi_clean = drop_zero_rows(aqi, ['pm25'])  # example
tmd_clean = tmd.copy()
aqi_clean = aqi.copy()

Cleaning sensor_data zeros...
  Zero-clean on ['pm25', 'temp', 'humidity', 'gas_co']: dropped 188 rows (30.9%) → 421 remaining
421


## 6 · Sort all tables by timestamp

In [27]:
print('Sorting by timestamp...')
sensor_sorted = sort_by_ts(sensor_clean)
tmd_sorted    = sort_by_ts(tmd_clean)
aqi_sorted    = sort_by_ts(aqi_clean)

print(f'  sensor range: {sensor_sorted[TS_COL].min()} → {sensor_sorted[TS_COL].max()}')
print(f'  tmd    range: {tmd_sorted[TS_COL].min()} → {tmd_sorted[TS_COL].max()}')
print(f'  aqi    range: {aqi_sorted[TS_COL].min()} → {aqi_sorted[TS_COL].max()}')

Sorting by timestamp...
  sensor range: 2026-03-31 00:55:18 → 2026-04-09 08:25:33
  tmd    range: 2026-03-31 10:00:00 → 2026-04-17 19:00:00
  aqi    range: 2026-03-31 01:00:00 → 2026-04-17 20:00:00


## 7 · Merge with `merge_asof` (nearest timestamp)

`merge_asof` is an ordered, tolerance-based join:
- Keeps **all** primary (`sensor_data`) rows.
- For each sensor row, finds the **nearest** TMD/AQI reading (past or future).
- Overlapping column names get `_tmd` / `_aqi` suffixes to avoid conflicts.

In [28]:
# ── Suffix overlapping columns before merge ───────────────────────────────────
# Columns shared between sensor and tmd (e.g., temp, humidity, lat, lon)
# will get a _tmd / _aqi suffix so you can compare them later.

tmd_merge = suffix_cols(tmd_sorted, '_tmd', exclude=[TS_COL])
aqi_merge = suffix_cols(aqi_sorted, '_aqi', exclude=[TS_COL])

print('Columns after suffix:')
print('  tmd:', list(tmd_merge.columns))
print('  aqi:', list(aqi_merge.columns))

Columns after suffix:
  tmd: ['ts', 'temp_tmd', 'humidity_tmd', 'rainfall_tmd', 'lat_tmd', 'lon_tmd']
  aqi: ['ts', 'pm25_aqi', 'pm10_aqi', 'lat_aqi', 'lon_aqi']


In [29]:
# ── Merge 1: sensor ← nearest TMD ────────────────────────────────────────────
# tolerance=None means no time limit on how far the nearest point can be.
# Set e.g. tolerance=pd.Timedelta('4h') to reject matches > 4 hours apart.

merged_tmd = pd.merge_asof(
    sensor_sorted,
    tmd_merge,
    on=TS_COL,
    direction='nearest',
    # tolerance=pd.Timedelta('4h'),  # uncomment to cap the match window
)
print(f'After TMD merge: {len(merged_tmd):,} rows')

After TMD merge: 421 rows


In [30]:
# ── Merge 2: (sensor+TMD) ← nearest AQI ─────────────────────────────────────

merged_all = pd.merge_asof(
    merged_tmd,
    aqi_merge,
    on=TS_COL,
    direction='nearest',
    # tolerance=pd.Timedelta('2h'),  # uncomment to cap the match window
)
print(f'After AQI merge: {len(merged_all):,} rows')

After AQI merge: 421 rows


## 8 · Post-merge inspection

In [31]:
print('=== Merged dataframe ===' )
display(merged_all.head(5))

print('\nShape:', merged_all.shape)
print('\nColumns:', list(merged_all.columns))

print('\nNull counts (should be low for merge columns):')
display(merged_all.isnull().sum()[merged_all.isnull().sum() > 0])

=== Merged dataframe ===


,ts,temp,humidity,pm25,pm10,gas_co,place,lat,lon,temp_tmd,humidity_tmd,rainfall_tmd,lat_tmd,lon_tmd,pm25_aqi,pm10_aqi,lat_aqi,lon_aqi
0,2026-03-31 00:55:18,29.0000,58,13,10,282,inside,13.8463,100.5680,32.5000,58.0000,0.0000,13.1617,100.8020,63,16,13.8476,100.5790
1,2026-03-31 00:55:35,27.0000,45,16,12,277,inside,13.8463,100.5680,32.5000,58.0000,0.0000,13.1617,100.8020,63,16,13.8476,100.5790
2,2026-03-31 01:05:35,26.0000,43,16,12,291,inside,13.8463,100.5680,32.5000,58.0000,0.0000,13.1617,100.8020,63,16,13.8476,100.5790
3,2026-03-31 01:15:35,28.0000,44,15,12,289,inside,13.8463,100.5680,32.5000,58.0000,0.0000,13.1617,100.8020,63,16,13.8476,100.5790
4,2026-03-31 01:25:36,29.0000,47,15,12,304,inside,13.8463,100.5680,32.5000,58.0000,0.0000,13.1617,100.8020,63,16,13.8476,100.5790



Shape: (421, 18)

Columns: ['ts', 'temp', 'humidity', 'pm25', 'pm10', 'gas_co', 'place', 'lat', 'lon', 'temp_tmd', 'humidity_tmd', 'rainfall_tmd', 'lat_tmd', 'lon_tmd', 'pm25_aqi', 'pm10_aqi', 'lat_aqi', 'lon_aqi']

Null counts (should be low for merge columns):


Series([], dtype: int64)

In [32]:
# Quick sanity: how far off are the nearest TMD / AQI matches?
if 'ts' in sensor_sorted.columns:
    merged_all['_tmd_lag_min'] = (merged_all[TS_COL] - tmd_merge[TS_COL].reindex(merged_all.index)).abs() / pd.Timedelta('1min')

print('Descriptive stats:')
display(merged_all.describe())

Descriptive stats:


,ts,temp,humidity,pm25,pm10,gas_co,lat,lon,temp_tmd,humidity_tmd,rainfall_tmd,lat_tmd,lon_tmd,pm25_aqi,pm10_aqi,lat_aqi,lon_aqi,_tmd_lag_min
count,421,421.0000,421.0000,421.0000,421.0000,421.0000,421.0000,421.0000,421.0000,421.0000,421.0000,421.0000,421.0000,421.0000,421.0000,421.0000,421.0000,139.0000
mean,2026-04-04 19:35:16.836104704,30.6651,51.6152,31.3302,21.7031,218.7411,13.8463,100.5680,30.9166,70.4181,0.0000,13.6101,100.5966,95.4442,33.4846,13.8476,100.5790,12189.3602
min,2026-03-31 00:55:18,25.0000,35.0000,9.0000,7.0000,138.0000,13.8463,100.5680,25.3000,45.0000,0.0000,12.7350,99.7924,59.0000,15.0000,13.8476,100.5790,544.7000
25%,2026-04-01 02:46:40,27.0000,46.0000,24.0000,16.0000,186.0000,13.8463,100.5680,28.9000,58.0000,0.0000,13.7069,100.5600,80.0000,24.0000,13.8476,100.5790,6459.2000
50%,2026-04-06 23:55:29,28.0000,51.0000,32.0000,22.0000,218.0000,13.8463,100.5680,30.4000,73.0000,0.0000,13.7069,100.5680,97.0000,34.0000,13.8476,100.5790,12211.9000
75%,2026-04-08 01:18:53,35.0000,57.0000,39.0000,27.0000,252.0000,13.8463,100.5680,32.5000,82.0000,0.0000,13.7264,100.6050,111.0000,43.0000,13.8476,100.5790,17833.3500
max,2026-04-09 08:25:33,39.0000,73.0000,53.0000,36.0000,318.0000,13.8463,100.5680,37.2000,90.0000,0.0000,13.9192,101.1350,129.0000,51.0000,13.8476,100.5790,23443.0667
std,NaN,3.9808,7.3500,11.1181,7.0132,39.5084,0.0000,0.0000,2.7934,13.7244,0.0000,0.2769,0.2052,18.4814,10.0642,0.0000,0.0000,6618.1799


## 9 · Save output

In [33]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Drop internal helper columns before saving
final = merged_all.drop(columns=[c for c in merged_all.columns if c.startswith('_')], errors='ignore')

final.to_csv(OUTPUT_CSV, index=False)
print(f'Saved {len(final):,} rows → {OUTPUT_CSV}')
display(final.head())

Saved 421 rows → output\integrated_air_quality_data.csv


,ts,temp,humidity,pm25,pm10,gas_co,place,lat,lon,temp_tmd,humidity_tmd,rainfall_tmd,lat_tmd,lon_tmd,pm25_aqi,pm10_aqi,lat_aqi,lon_aqi
0,2026-03-31 00:55:18,29.0000,58,13,10,282,inside,13.8463,100.5680,32.5000,58.0000,0.0000,13.1617,100.8020,63,16,13.8476,100.5790
1,2026-03-31 00:55:35,27.0000,45,16,12,277,inside,13.8463,100.5680,32.5000,58.0000,0.0000,13.1617,100.8020,63,16,13.8476,100.5790
2,2026-03-31 01:05:35,26.0000,43,16,12,291,inside,13.8463,100.5680,32.5000,58.0000,0.0000,13.1617,100.8020,63,16,13.8476,100.5790
3,2026-03-31 01:15:35,28.0000,44,15,12,289,inside,13.8463,100.5680,32.5000,58.0000,0.0000,13.1617,100.8020,63,16,13.8476,100.5790
4,2026-03-31 01:25:36,29.0000,47,15,12,304,inside,13.8463,100.5680,32.5000,58.0000,0.0000,13.1617,100.8020,63,16,13.8476,100.5790


---
## Appendix: Load from MySQL directly (alternative to CSV)

Uncomment and use this cell instead of §3 if you want to pull straight from the DB.

In [34]:
# from sqlalchemy import create_engine
# from dotenv import load_dotenv
#
# load_dotenv('../backend/.env')   # reads DB_HOST, DB_PORT, etc.
#
# DB_URL = (
#     f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
#     f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
# )
# engine = create_engine(DB_URL)
#
# sensor = pd.read_sql('SELECT * FROM sensor_data ORDER BY ts', engine)
# tmd    = pd.read_sql('SELECT * FROM tmd_data   ORDER BY ts', engine)
# aqi    = pd.read_sql('SELECT * FROM aqi_data   ORDER BY ts', engine)
# engine.dispose()
# print('Loaded directly from MySQL.')